# AI in Additive Manufacturing — In-Class Activities
## Duke University | AI in Manufacturing
## Designed by Dr. Jacob Peloquin

| Activity | Topic | Key concept |
|---|---|---|
| **1** | In-situ monitoring & defect detection | Which sensors matter? On/off sensor selection |
| **2** | Defect forecasting | Predicting risk in future layers from planned parameters |
| **3** | In-situ control & decisions | Threshold, cost tradeoff, intervention decisions |


### ▶ Run this setup cell first (all activities depend on it)

In [ ]:
import sys, subprocess, warnings
warnings.filterwarnings('ignore')
subprocess.run([sys.executable,'-m','pip','install','-q','scikit-learn','seaborn',
                '--break-system-packages'], capture_output=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({'figure.dpi':110, 'axes.spines.top':False,
                     'axes.spines.right':False, 'axes.titlepad':12})

DATA_PATH = './'
# Google Colab — clone your GitHub repo first, then set DATA_PATH:
# !git clone https://github.com/YOUR_USER/YOUR_REPO.git
# DATA_PATH = './YOUR_REPO/'

# ── Load Activity 1 data ──────────────────────────────────────────────
df1       = pd.read_csv(DATA_PATH + 'act1_sensor_data.csv')
df1_melt  = pd.read_csv(DATA_PATH + 'act1_meltpool_data.csv')
df1_opt   = pd.read_csv(DATA_PATH + 'act1_optical_data.csv')

N1, IMG = len(df1), 64
print('Caching Act1 images...', end=' ')
MELT1 = np.zeros((N1,IMG,IMG)); OPT1 = np.zeros((N1,IMG,IMG))
for L in range(1,N1+1):
    MELT1[L-1] = df1_melt[df1_melt.layer==L].pivot(index='row',columns='col',values='temp').values
    OPT1[L-1]  = df1_opt[df1_opt.layer==L].pivot(index='row',columns='col',values='reflectance').values
print('done.')

FEAT_COLS = ['o2_level','melt_mean','melt_std','optical_mean','optical_std']
X1 = df1[FEAT_COLS].values
y1 = df1['defect_present'].values
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
FEATS_ACT1 = {'O2':['o2_level'], 'Melt':['melt_mean','melt_std'],
              'Optical':['optical_mean','optical_std']}

# ── Load Activity 2/3 data ────────────────────────────────────────────
df23 = pd.read_csv(DATA_PATH + 'act23_data.csv')

# Physics-based risk formula (transparent, parameter-sensitive)
def physics_risk(o2, power, speed, hatch, melt_std, noise=0.0):
    """
    Continuous defect risk score (0.05–0.94) from process physics.
      Porosity   ← high O2 OR low energy density (low power / high speed)
      Spatter    ← excess energy density (high power / low speed)
      Super-elev ← high melt pool spatial variance (melt_std)
    """
    E     = power / (speed * hatch)
    E_nom = 200.0  / (1200.0 * 0.110)
    deficit = max(0, (E_nom - E) / E_nom)   # energy deficit  → porosity
    excess  = max(0, (E - E_nom) / E_nom)   # energy excess   → spatter
    o2_r    = max(0, (o2 - 0.025) / 0.08)  # O2 contamination
    melt_r  = max(0, (melt_std - 60) / 140) # hot-spot variance
    raw = 1.8*o2_r + 1.5*deficit + 1.3*excess + 0.9*melt_r + noise
    return float(np.clip(1.0/(1.0+np.exp(-3.2*(raw-0.50))), 0.05, 0.94))

print(f'Act1: {N1} layers  |  defect rate: {y1.mean():.0%}')
print(f'Act23: {len(df23)} layers  |  risk range: {df23.risk_score.min():.2f}–{df23.risk_score.max():.2f}')


# Activity 1 — In-Situ Monitoring & Defect Detection

**Build scenario:** Three parts are being printed simultaneously:
- **Hollow cylinder** (layers 1–20)  ·  **Solid cube** (layers 1–40)  ·  **Gyroid cube** (layers 1–38)

Three sensor streams are recorded every layer:

| # | Sensor | Defect link |
|---|---|---|
| **S1** | Chamber O₂ level | PRIMARY for **porosity** (energy deficit + O₂ contamination) |
| **S2** | Melt pool temperature | PRIMARY for **spatter** (high mean) and **super-elevation** (high std) |
| **S3** | Post-melt optical image | Weak secondary signal — visually intuitive but not the strongest predictor |


### A1 · Cell 1 — Sensor Stream Visualization
**Set `LAYER` and `SIGNAL`, then re-run.**

In [ ]:
# ┌──────────────────────────────────────────────────────────┐
# │  LAYER  = 1 to 40  │  SIGNAL = 1 · O₂ (temporal)       │
# │                     │           2 · Melt pool (spatial)  │
# │                     │           3 · Optical image         │
# └──────────────────────────────────────────────────────────┘
LAYER  = 10
SIGNAL = 2

if SIGNAL == 1:
    fig, ax = plt.subplots(figsize=(12, 3.2))
    ax.plot(df1.layer, df1.o2_level*100, 'o-', color='steelblue', ms=4, lw=1.5)
    ax.axvline(LAYER, color='red', lw=1.5, ls='--', label=f'Layer {LAYER}')
    ax.scatter(df1[df1.defect_present==1].layer,
               df1[df1.defect_present==1].o2_level*100,
               color='tomato', s=65, zorder=5, label='Defect layer')
    ax.axhline(5.0, color='orange', lw=1, ls=':', label='Contamination threshold (5%)')
    ax.set_xlabel('Layer'); ax.set_ylabel('O₂ (%)')
    ax.set_title('Signal 1 — Chamber O₂ Level', fontweight='bold')
    ax.legend(fontsize=8); plt.tight_layout(); plt.show()

elif SIGNAL == 2:
    layer_means = [MELT1[L-1][MELT1[L-1]>400].mean()
                   if (MELT1[L-1]>400).any() else 110 for L in range(1,N1+1)]
    fig, (a1,a2) = plt.subplots(1,2,figsize=(13,4.8))
    a1.plot(range(1,N1+1), layer_means, 'o-', color='tomato', ms=4, lw=1.5)
    a1.axvline(LAYER, color='navy', lw=1.5, ls='--', label=f'Layer {LAYER}')
    a1.set_xlabel('Layer'); a1.set_ylabel('Mean melt-pool temp (°C)')
    a1.set_title('Mean melt-pool temp per layer', fontweight='bold')
    a1.legend(fontsize=8)
    im = a2.imshow(MELT1[LAYER-1], cmap='inferno', origin='upper')
    plt.colorbar(im, ax=a2, label='Temp (°C)')
    r,c = np.ogrid[:IMG,:IMG]; dist = np.sqrt((r-32)**2+(c-10)**2)
    ring = ((dist<=10.5)&(dist>=4.5)).astype(float); ring[ring==0]=np.nan
    a2.contour(ring, levels=[0.5], colors=['cyan'], linewidths=1.3)
    a2.add_patch(plt.Rectangle((23.5,20.5),20,22,fill=False,ec='lime',lw=1.3))
    if LAYER<=38: a2.add_patch(plt.Rectangle((46.5,9.5),16,44,fill=False,ec='yellow',lw=1.3))
    a2.set_title(f'Melt-pool map — Layer {LAYER}  (cyan=cylinder, green=cube, yellow=gyroid)',
                 fontsize=8.5, fontweight='bold')
    a2.set_xlabel('Col (px)'); a2.set_ylabel('Row (px)')
    plt.tight_layout(); plt.show()
    row = df1[df1.layer==LAYER].iloc[0]
    print(f'Layer {LAYER}: melt_mean={row.melt_mean:.0f}°C  melt_std={row.melt_std:.0f}  '
          f'defect={row.defect_type}')

elif SIGNAL == 3:
    fig, ax = plt.subplots(figsize=(5.5,5.5))
    im = ax.imshow(OPT1[LAYER-1], cmap='gray', vmin=0, vmax=1, origin='upper')
    plt.colorbar(im, ax=ax, label='Reflectance (0=powder, 1=melted)')
    ax.set_title(f'Signal 3 — Optical Image, Layer {LAYER}', fontweight='bold')
    ax.set_xlabel('Col (px)'); ax.set_ylabel('Row (px)')
    plt.tight_layout(); plt.show()


### A1 · Cell 2 — Defect Type Examples

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(13,4.2))
for ax,(fpath,name,desc) in zip(axes,[
    (DATA_PATH+'defect_porosity.jpg',       'Porosity',
     'O₂ contamination or low energy\n→ dark subsurface voids'),
    (DATA_PATH+'defect_spatter.jpg',         'Spatter',
     'Excess energy density\n→ bright ejected metal particles'),
    (DATA_PATH+'defect_super_elevation.jpg', 'Super-elevation',
     'Localised hot spots\n→ bright raised ridge'),
]):
    try:    ax.imshow(mpimg.imread(fpath))
    except: ax.text(0.5,0.5,f'[{name}]\nfile not found',ha='center',va='center',transform=ax.transAxes)
    ax.set_title(f'{name}\n{desc}', fontsize=9.5, fontweight='bold', linespacing=1.5, pad=10)
    ax.axis('off')
plt.suptitle('Defect Type Examples — Post-Melt Optical Appearance',
             fontsize=11, fontweight='bold', y=1.03)
plt.tight_layout(); plt.show()


### A1 · Cell 3 — Defect Detection Model *(Attempt 1)*

The model is a **weighted ensemble of three sub-models** — one trained per sensor.  
Each sub-model only uses that sensor's features. The final prediction is a weighted  
average of the three sub-model outputs.

**Start with `USE_OPTICAL = True` and the others `False` — this is the intuitive  
first guess (optical image looks most informative). See the F1 result, then  
use Cells 4 & 5 to find a better combination.**


In [ ]:
# ┌──────────────────────────────────────────────────────────────────┐
# │  Set each sensor True (include) or False (exclude), re-run      │
# └──────────────────────────────────────────────────────────────────┘
USE_O2      = False   # Sensor 1 · Chamber O₂
USE_MELT    = False   # Sensor 2 · Melt Pool Temperature
USE_OPTICAL = False    # Sensor 3 · Post-Melt Optical Image

# ── Build ensemble from selected sensors ──────────────────────────────────────
selection = {'O2': USE_O2, 'Melt': USE_MELT, 'Optical': USE_OPTICAL}
active    = {k:v for k,v in FEATS_ACT1.items() if selection[k]}

if not active:
    print("⚠  Select at least one sensor.");
else:
    # Train each sub-model with CV, average probabilities
    fold_f1s = []
    for tr_idx, te_idx in CV.split(X1, y1):
        probs = np.zeros(len(te_idx))
        n_active = len(active)
        for name, cols in active.items():
            Xtr = df1.iloc[tr_idx][cols].values
            Xte = df1.iloc[te_idx][cols].values
            sc  = StandardScaler().fit(Xtr)
            clf = LogisticRegression(max_iter=500, class_weight='balanced', random_state=42)
            clf.fit(sc.transform(Xtr), y1[tr_idx])
            probs += clf.predict_proba(sc.transform(Xte))[:,1] / n_active
        fold_f1s.append(f1_score(y1[te_idx], (probs>0.5).astype(int), zero_division=0))
    cv_f1_v1 = float(np.mean(fold_f1s))

    fig, ax = plt.subplots(figsize=(6,3.5))
    sensor_labels = list(FEATS_ACT1.keys())
    colors_bar = ['#2ecc71' if selection[s] else '#bdc3c7' for s in sensor_labels]
    bars = ax.bar(sensor_labels, [1 if selection[s] else 0 for s in sensor_labels],
                  color=colors_bar, edgecolor='white', width=0.5)
    ax.set_ylim(0,1.3); ax.set_yticks([]);
    ax.set_title(f'Sensors used  ·  CV F1 = {cv_f1_v1:.3f}', fontweight='bold')
    for bar, s in zip(bars, sensor_labels):
        label = '✓ ON' if selection[s] else '✗ OFF'
        ax.text(bar.get_x()+bar.get_width()/2, 1.05, label,
                ha='center', fontsize=10, fontweight='bold',
                color='#27ae60' if selection[s] else '#7f8c8d')
    plt.tight_layout(); plt.show()
    print(f'Sensors active: {list(active.keys())}')
    print(f'CV F1 (5-fold): {cv_f1_v1:.3f}')


### A1 · Cell 4 — Feature Correlation with Defect Presence
Which sensor features correlate most strongly with defects?  
**Use this, alongside Cell 5, to choose better sensors in Cell 6.**


In [ ]:
corr = df1[FEAT_COLS+['defect_present']].corr()['defect_present'].drop('defect_present')
fig, ax = plt.subplots(figsize=(8,3.8))
colors_bar = ['#e74c3c' if v>0 else '#3498db' for v in corr.values]
bars = ax.bar(range(len(FEAT_COLS)), corr.values, color=colors_bar, edgecolor='white', width=0.6)
ax.set_xticks(range(len(FEAT_COLS)))
ax.set_xticklabels(['O₂ level\n(S1)','Melt mean\n(S2)','Melt std\n(S2)',
                     'Optical mean\n(S3)','Optical std\n(S3)'], fontsize=9)
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Pearson Correlation with Defect Presence')
ax.set_title('Feature Correlation with Defect Presence', fontweight='bold')
# Sensor group shading
for span, color, label in [((-0.5,0.5),'#8e44ad','S1: O₂'),
                             ((0.5,2.5),'#e67e22','S2: Melt'),
                             ((2.5,4.5),'#27ae60','S3: Optical')]:
    ax.axvspan(*span, alpha=0.07, color=color)
    ax.text(sum(span)/2, ax.get_ylim()[0]*0.85, label,
            ha='center', fontsize=8, color=color, fontweight='bold')
for bar, v in zip(bars, corr.values):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.015 if v>=0 else v-0.04,
            f'{v:.2f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout(); plt.show()


### A1 · Cell 5 — Model Reliance (Automated Permutation Test)
Each sensor is shuffled randomly, the model retrained, and the F1 drop measured.  
**Bigger drop = model relied on that sensor more.**


In [ ]:
base_f1s = []
for tr_idx, te_idx in CV.split(X1, y1):
    probs = np.zeros(len(te_idx))
    for name, cols in FEATS_ACT1.items():
        Xtr=df1.iloc[tr_idx][cols].values; Xte=df1.iloc[te_idx][cols].values
        sc=StandardScaler().fit(Xtr)
        clf=LogisticRegression(max_iter=500,class_weight='balanced',random_state=42)
        clf.fit(sc.transform(Xtr),y1[tr_idx])
        probs+=clf.predict_proba(sc.transform(Xte))[:,1]/3
    base_f1s.append(f1_score(y1[te_idx],(probs>0.5).astype(int),zero_division=0))
base_f1 = float(np.mean(base_f1s))

reliance = {}
np.random.seed(77)
for sensor_name, cols in FEATS_ACT1.items():
    fold_f1s = []
    for tr_idx, te_idx in CV.split(X1, y1):
        probs = np.zeros(len(te_idx))
        for name, fcols in FEATS_ACT1.items():
            Xtr=df1.iloc[tr_idx][fcols].values.copy()
            Xte=df1.iloc[te_idx][fcols].values.copy()
            if name == sensor_name:   # scramble this sensor
                perm = np.random.permutation(len(Xtr))
                Xtr  = Xtr[perm]
            sc=StandardScaler().fit(Xtr)
            clf=LogisticRegression(max_iter=500,class_weight='balanced',random_state=42)
            clf.fit(sc.transform(Xtr),y1[tr_idx])
            probs+=clf.predict_proba(sc.transform(Xte))[:,1]/3
        fold_f1s.append(f1_score(y1[te_idx],(probs>0.5).astype(int),zero_division=0))
    reliance[sensor_name] = max(0, base_f1 - float(np.mean(fold_f1s)))

fig, ax = plt.subplots(figsize=(7,3.2))
names, vals = list(reliance.keys()), list(reliance.values())
cmap_r = plt.cm.Oranges(np.array(vals)/max(max(vals),0.001)*0.75+0.25)
bars   = ax.barh(names, vals, color=cmap_r, edgecolor='white')
ax.set_xlabel('F1 Drop When Sensor Scrambled  (higher = model relied on it more)')
ax.set_title(f'Model Reliance  (baseline equal-weight CV F1 = {base_f1:.3f})', fontweight='bold')
for bar, v in zip(bars, vals):
    ax.text(v+0.003, bar.get_y()+bar.get_height()/2,
            f'{v:.3f}', va='center', fontsize=9, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Most relied-on sensor: {names[np.argmax(vals)]}  (drop={max(vals):.3f})')
print()
print('Now use Cells 4 & 5 together to choose which sensors to include in Cell 6.')


### A1 · Cell 6 — Detection Model *(Attempt 2 — informed selection)*
Using the correlation matrix and reliance analysis, choose which sensors to include.  
**Compare your result to Attempt 1.**


In [ ]:
# ┌──────────────────────────────────────────────────────────────────┐
# │  Update based on Cells 4 & 5 — which sensors actually matter?  │
# └──────────────────────────────────────────────────────────────────┘
USE_O2      = False    # ← try changing
USE_MELT    = False   # ← try changing
USE_OPTICAL = False   # ← try changing

selection2 = {'O2': USE_O2, 'Melt': USE_MELT, 'Optical': USE_OPTICAL}
active2    = {k:v for k,v in FEATS_ACT1.items() if selection2[k]}

if not active2:
    print("⚠  Select at least one sensor.")
else:
    fold_f1s2 = []
    for tr_idx, te_idx in CV.split(X1, y1):
        probs2 = np.zeros(len(te_idx))
        n2     = len(active2)
        for name, cols in active2.items():
            Xtr=df1.iloc[tr_idx][cols].values; Xte=df1.iloc[te_idx][cols].values
            sc=StandardScaler().fit(Xtr)
            clf=LogisticRegression(max_iter=500,class_weight='balanced',random_state=42)
            clf.fit(sc.transform(Xtr),y1[tr_idx])
            probs2+=clf.predict_proba(sc.transform(Xte))[:,1]/n2
        fold_f1s2.append(f1_score(y1[te_idx],(probs2>0.5).astype(int),zero_division=0))
    cv_f1_v2 = float(np.mean(fold_f1s2))

    fig, ax = plt.subplots(figsize=(6,3.8))
    f1s  = [cv_f1_v1, cv_f1_v2]
    cols_bar = ['#95a5a6','#2ecc71' if cv_f1_v2>cv_f1_v1 else '#e74c3c']
    bars = ax.bar(['Attempt 1','Attempt 2'], f1s, color=cols_bar,
                  edgecolor='white', width=0.4)
    ax.set_ylim(0,1); ax.set_ylabel('Cross-validated F1 Score (5-fold)')
    ax.set_title('Detection Performance: Before vs After Analysis', fontweight='bold')
    delta = cv_f1_v2 - cv_f1_v1
    ax.text(1, cv_f1_v2+0.03, f'{delta:+.3f}', ha='center', fontsize=12,
            fontweight='bold', color='#27ae60' if delta>0 else '#e74c3c')
    for bar, v in zip(bars, f1s):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.015, f'{v:.3f}',
                ha='center', fontsize=11, fontweight='bold')
    plt.tight_layout(); plt.show()
    print(f'Attempt 1: {cv_f1_v1:.3f}  Attempt 2: {cv_f1_v2:.3f}  Δ={delta:+.3f}')
    print(f'Sensors in Attempt 2: {list(active2.keys())}')


# Activity 2 — Defect Forecasting

**Scenario:** You are at **layer 20** of a 50-layer build.  
Layers 1–20 are complete — sensor readings are available.  
Layers 21–50 are not yet printed — only the **planned machine parameters** are known.

**Goal:** Use a physics-based risk model to forecast the **continuous defect risk score**  
(0 = definitely safe, 1 = definite defect) for each upcoming layer.

The risk score is driven by:
- **O₂ contamination** → porosity risk  
- **Energy density** (power / speed / hatch) → too low = porosity, too high = spatter  
- **Melt pool variance** → super-elevation risk

**Threshold:** a layer is "flagged for intervention" when its risk score exceeds `THRESHOLD`.  
Adjusting the threshold controls the sensitivity — lower threshold = more interventions.


### A2 · Cell 1 — Build History (Layers 1–20)

In [ ]:
hist23 = df23[df23.layer<=20]
fore23 = df23[df23.layer>20]

fig, axes = plt.subplots(1,3,figsize=(14,3.8))

# O2 history
axes[0].plot(hist23.layer, hist23.o2_level*100,'o-',color='steelblue',ms=5,lw=1.5)
axes[0].scatter(hist23[hist23.risk_score>0.15].layer,
                hist23[hist23.risk_score>0.15].o2_level*100,
                color='tomato',s=75,zorder=5,label='Risk>0.15')
axes[0].axhline(5.0,color='orange',lw=1,ls=':',label='O₂ threshold')
axes[0].set_xlabel('Layer'); axes[0].set_ylabel('O₂ (%)')
axes[0].set_title('O₂ Level — layers 1–20',fontweight='bold')
axes[0].legend(fontsize=8)

# Power history
axes[1].bar(hist23.layer, hist23.laser_power_W,
            color=['tomato' if p>215 or p<185 else 'steelblue' for p in hist23.laser_power_W],
            edgecolor='white')
axes[1].axhline(215,color='tomato',lw=1,ls=':',label='High-risk threshold')
axes[1].axhline(185,color='tomato',lw=1,ls=':')
axes[1].axhline(200,color='gray',lw=1,ls='--',alpha=0.5,label='Nominal 200W')
axes[1].set_xlabel('Layer'); axes[1].set_ylabel('Laser Power (W)')
axes[1].set_title('Laser Power — layers 1–20',fontweight='bold')
axes[1].legend(fontsize=8)

# Risk score history (ground truth)
axes[2].bar(hist23.layer, hist23.risk_score,
            color=['tomato' if r>0.15 else 'steelblue' for r in hist23.risk_score],
            edgecolor='white')
axes[2].axhline(0.15,color='red',lw=1.5,ls=':',label='Risk threshold 0.15')
axes[2].set_xlabel('Layer'); axes[2].set_ylabel('Risk Score (0–1)')
axes[2].set_title('Observed Risk Score — layers 1–20',fontweight='bold')
axes[2].legend(fontsize=8)

plt.suptitle('Build History — Completed Layers 1–20',fontsize=11,fontweight='bold',y=1.04)
plt.tight_layout(); plt.show()

n_high = (hist23.risk_score>0.15).sum()
print(f'Layers 1-20: {n_high} high-risk layers (score > 0.15)')
print(f'High-risk layers: {list(hist23[hist23.risk_score>0.15].layer.values)}')
print(f'Upcoming forecast window: layers 21–50')


### A2 · Cell 2 — Defect Risk Forecast (Layers 21–50)

The **physics-based risk model** forecasts a continuous risk score for each upcoming layer  
using the planned machine parameters and expected melt pool conditions.

**Set `THRESHOLD` and re-run.** Layers shaded red have risk above the threshold.  
More layers are shaded as you lower the threshold.


In [ ]:
# ┌──────────────────────────────────────────────────────────┐
# │  THRESHOLD: layers with risk ≥ this are flagged          │
# └──────────────────────────────────────────────────────────┘
THRESHOLD = 0.15

# Compute forecast risk using physics formula on planned parameters
np.random.seed(42)
fore_risk = []
for _, row in fore23.iterrows():
    noise = np.random.normal(0, 0.04)
    r = physics_risk(row.o2_level, row.laser_power_W, row.scan_speed_mms,
                     row.hatch_spacing, row.melt_std, noise=noise)
    fore_risk.append(r)
fore23 = fore23.copy()
fore23['forecast_risk'] = fore_risk
fore23['flagged'] = (fore23.forecast_risk >= THRESHOLD).astype(int)
fore23['true_defect'] = fore23['true_defect'].astype(int)

# Also get forecast risk on history for continuity
hist_risk = []
for _, row in hist23.iterrows():
    hist_risk.append(row.risk_score)

fig, ax = plt.subplots(figsize=(13,4.5))
ax.fill_between(hist23.layer, 0, 1, alpha=0.06, color='steelblue', label='Observed layers')
ax.fill_between(fore23.layer, 0, 1, alpha=0.06, color='silver',    label='Forecast window')

ax.plot(hist23.layer, hist23.risk_score, 'o-', color='steelblue', ms=5, lw=1.5, label='Observed risk')
ax.plot(fore23.layer, fore23.forecast_risk, 's--', color='tomato', ms=6, lw=2, label='Forecasted risk')
ax.axhline(THRESHOLD, color='red', lw=1.5, ls=':', label=f'Threshold = {THRESHOLD}')

for fl in fore23[fore23.flagged==1].layer:
    ax.axvspan(fl-0.4, fl+0.4, alpha=0.25, color='tomato')

ax.axvline(20.5, color='gray', lw=1.2, ls='--')
ax.text(21.0, 0.93, '→ Forecast', fontsize=8.5, color='gray')
ax.set_xlabel('Layer'); ax.set_ylabel('Defect Risk Score (0 = safe, 1 = certain defect)')
ax.set_title(f'Defect Risk Forecast — Layers 21–50  |  Threshold = {THRESHOLD}',
             fontweight='bold')
ax.set_ylim(0,1); ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

flagged = fore23[fore23.flagged==1]
print(f'Forecast: {len(fore23)} future layers  |  Flagged at threshold {THRESHOLD}: {len(flagged)}')
for _, row in flagged.iterrows():
    print(f'  Layer {int(row.layer):2d}: risk={row.forecast_risk:.3f}  '
          f'power={row.laser_power_W:.0f}W  speed={row.scan_speed_mms:.0f}mm/s')


### A2 · Cell 3 — What-If: Override Planned Parameters

**How it works:** The risk score formula is:
```
risk ∝  1.8 × O₂ contamination  +  1.5 × energy deficit  +  1.3 × energy excess  +  0.9 × melt variance
where  energy density = laser_power / (scan_speed × hatch_spacing)
```

Override the planned parameters for **all forecast layers** and see how the forecast changes.  
This simulates updating the machine controller before printing those layers.

- **Low power (< ~185W) or high speed (> ~1280)** → energy deficit → porosity risk ↑  
- **High power (> ~215W) or low speed (< ~1160)** → energy excess → spatter risk ↑  
- **Nominal (~200W, ~1200mm/s)** → minimum risk zone


In [ ]:
# ┌──────────────────────────────────────────────────────────────────────────┐
# │  Override planned parameters for ALL forecast layers (21-50)            │
# │  Nominal: power=200W  speed=1200mm/s  hatch=0.110mm                     │
# │  Low power (<185W) + high speed (>1280) → porosity risk rises           │
# │  High power (>215W) + low speed (<1160) → spatter risk rises            │
# └──────────────────────────────────────────────────────────────────────────┘
NEW_LASER_POWER_W  = 100.0   # W
NEW_SCAN_SPEED_MMS = 1400.0  # mm/s
NEW_HATCH_SPACING  = 0.080   # mm

# ── Recompute risk with overridden parameters ─────────────────────────────────
# O2 and melt_std still come from the data (sensor readings depend on process)
np.random.seed(42)
new_risk = []
for _, row in fore23.iterrows():
    # O2 level: slightly higher when energy density is low (less effective purging)
    E_new  = NEW_LASER_POWER_W / (NEW_SCAN_SPEED_MMS * NEW_HATCH_SPACING)
    E_nom  = 200.0 / (1200.0 * 0.110)
    # Effective O2 tracks the original reading but shifts with energy deficit
    o2_adj = row.o2_level + max(0, (E_nom - E_new)/E_nom) * 0.025
    noise  = np.random.normal(0, 0.04)
    r = physics_risk(o2_adj, NEW_LASER_POWER_W, NEW_SCAN_SPEED_MMS,
                     NEW_HATCH_SPACING, row.melt_std, noise=noise)
    new_risk.append(r)

fore23_new = fore23.copy()
fore23_new['new_risk'] = new_risk

fig, ax = plt.subplots(figsize=(11,4.5))
x = fore23.layer.values
ax.plot(x, fore23.forecast_risk, 's--', color='tomato',   ms=6, lw=1.8,
        label='Original planned parameters')
ax.plot(x, new_risk,            'o-',  color='steelblue', ms=6, lw=1.8,
        label='Your overridden parameters')
ax.axhline(THRESHOLD, color='black', lw=1.2, ls=':', label=f'Threshold = {THRESHOLD}')
ax.fill_between(x, fore23.forecast_risk, new_risk, alpha=0.18, color='steelblue')
ax.set_xlabel('Layer'); ax.set_ylabel('Forecasted Defect Risk Score')
ax.set_title(
    f'What-If Analysis — Power={NEW_LASER_POWER_W:.0f}W  '
    f'Speed={NEW_SCAN_SPEED_MMS:.0f}mm/s  Hatch={NEW_HATCH_SPACING:.3f}mm',
    fontweight='bold')
ax.set_ylim(0,1); ax.legend(fontsize=8)

# Annotate energy density change
E_new  = NEW_LASER_POWER_W/(NEW_SCAN_SPEED_MMS*NEW_HATCH_SPACING)
E_nom  = 200.0/(1200.0*0.110)
E_pct  = (E_new-E_nom)/E_nom*100
ax.text(0.01, 0.97,
        f'Energy density: {E_pct:+.1f}% vs nominal  →  '
        f'{"Spatter risk ↑" if E_pct>5 else "Porosity risk ↑" if E_pct<-5 else "Near nominal ✓"}',
        transform=ax.transAxes, fontsize=9, va='top',
        bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', ec='orange', alpha=0.9))
plt.tight_layout(); plt.show()

orig_flagged = int((np.array(fore23.forecast_risk)>=THRESHOLD).sum())
new_flagged  = int((np.array(new_risk)>=THRESHOLD).sum())
print(f'Original plan:      {orig_flagged} layers flagged')
print(f'Overridden params:  {new_flagged} layers flagged  ({new_flagged-orig_flagged:+d})')
print(f'Energy density:     {E_pct:+.1f}% relative to nominal (200W / 1200mm/s / 0.110mm)')


# Activity 3 — In-Situ Control & Build Intervention

**Scenario:** Same build — you are at **layer 20**, and the Activity 2 forecast is in hand.  
For each high-risk layer you must choose one of three actions:

| Action | Effect | Cost |
|---|---|---|
| ✅ **Continue** | Accept the risk — defect may form | Potential defect cost |
| ⚙️ **Adjust** | Update parameters before printing that layer | ~\$400 per intervention |
| 🛑 **Stop** | Abort and restart the entire build | ~\$15,000 |

**Two key questions:**
1. *At what risk level should you intervene?* → Cost tradeoff analysis (Cell 2)  
2. *What action is best for each flagged layer?* → Decision matrix (Cell 3)


### A3 · Cell 1 — Build Risk Dashboard
**Set `THRESHOLD` and re-run.** Layers shaded red are flagged for intervention.  
Lower threshold = flag more layers (more false alarms but fewer missed defects).


In [ ]:
# ┌──────────────────────────────────────────────────────────┐
# │  THRESHOLD: risk ≥ this → flagged for intervention       │
# └──────────────────────────────────────────────────────────┘
THRESHOLD_A3 = 0.15

# Use the forecast from Activity 2 (fore23 with forecast_risk)
all_risk = pd.concat([
    hist23.assign(risk=hist23.risk_score, zone='observed'),
    fore23.assign(risk=fore23.forecast_risk, zone='forecast'),
]).sort_values('layer').reset_index(drop=True)

fig, axes = plt.subplots(1,2,figsize=(14,4.8))

# Left: timeline
ax = axes[0]
obs  = all_risk[all_risk.zone=='observed']
fore_d = all_risk[all_risk.zone=='forecast']
ax.fill_between(obs.layer,   0, 1, alpha=0.06, color='steelblue')
ax.fill_between(fore_d.layer, 0, 1, alpha=0.06, color='silver')
ax.plot(obs.layer,   obs.risk,   'o-',  color='steelblue', ms=4, lw=1.5, label='Observed')
ax.plot(fore_d.layer, fore_d.risk, 's--', color='tomato',   ms=5, lw=1.8, label='Forecasted')
ax.axhline(THRESHOLD_A3, color='red', lw=1.5, ls=':', label=f'Threshold = {THRESHOLD_A3}')
ax.axvline(20.5, color='gray', lw=1.2, ls='--')
ax.text(21.2, 0.93, '→ Forecast', fontsize=8.5, color='gray')
for _, row in fore_d[fore_d.risk>=THRESHOLD_A3].iterrows():
    ax.axvspan(row.layer-0.4, row.layer+0.4, alpha=0.22, color='tomato')
ax.set_xlabel('Layer'); ax.set_ylabel('Defect Risk Score')
ax.set_title('Build Risk Timeline', fontweight='bold')
ax.set_ylim(0,1); ax.legend(fontsize=8)

# Right: bar chart
ax2 = axes[1]
bar_colors = []
for _, row in all_risk.iterrows():
    if row.zone=='observed':
        bar_colors.append('tomato' if row.risk>0.45 else 'steelblue')
    else:
        bar_colors.append('salmon' if row.risk>=THRESHOLD_A3 else 'lightsteelblue')
ax2.bar(all_risk.layer, all_risk.risk, color=bar_colors, edgecolor='white', width=0.7)
ax2.axhline(THRESHOLD_A3, color='red', lw=1.5, ls=':')
ax2.axvline(20.5, color='gray', lw=1.2, ls='--')
ax2.set_xlabel('Layer'); ax2.set_ylabel('Risk Score')
ax2.set_title('Risk Score per Layer', fontweight='bold'); ax2.set_ylim(0,1)
handles = [mpatches.Patch(color='tomato',      label='Observed high-risk'),
           mpatches.Patch(color='steelblue',   label='Observed safe'),
           mpatches.Patch(color='salmon',      label='Forecast: flagged'),
           mpatches.Patch(color='lightsteelblue', label='Forecast: safe')]
ax2.legend(handles=handles, fontsize=7.5, ncol=2)
plt.tight_layout(); plt.show()

flagged_a3 = fore_d[fore_d.risk>=THRESHOLD_A3]
print(f'Flagged layers ({len(flagged_a3)}): {list(flagged_a3.layer.values)}')


### A3 · Cell 2 — Cost Tradeoff vs Threshold

**How the curve works:**
- **Low threshold** (flag everything) → many **false alarms** (interventions on safe layers) → high false-alarm cost  
- **High threshold** (flag very little) → many **missed defects** (defects that slip through) → high defect cost  
- **Optimal threshold** = the crossing point where total cost is minimised

Adjust costs below to see how the optimal threshold shifts.


In [ ]:
# ┌──────────────────────────────────────────────────────────────────┐
# │  Adjust cost assumptions and re-run to see the tradeoff shift   │
# └──────────────────────────────────────────────────────────────────┘
COST_FALSE_ALARM   =  100    # $ — unnecessary intervention on a safe layer
COST_MISSED_DEFECT = 100    # $ — defect reaches post-process inspection

# Ground truth: forecast layers where risk > 0.50 are "true defect" layers
# (Threshold 0.50 on the continuous forecast score as the gold-standard)
true_def = fore_d['true_defect'].values  # ground truth defect labels
fore_scores = fore_d['risk'].values

thresholds = np.linspace(0.05, 0.95, 300)
fp_costs, fn_costs, total_costs = [], [], []

for t in thresholds:
    preds = (fore_scores >= t).astype(int)
    fp = ((preds==1) & (true_def==0)).sum()
    fn = ((preds==0) & (true_def==1)).sum()
    fp_costs.append(fp  * COST_FALSE_ALARM)
    fn_costs.append(fn  * COST_MISSED_DEFECT)
    total_costs.append(fp*COST_FALSE_ALARM + fn*COST_MISSED_DEFECT)

best_t = float(thresholds[np.argmin(total_costs)])

fig, ax = plt.subplots(figsize=(12,4.5))
ax.plot(thresholds, fp_costs,   color='steelblue', lw=1.8,
        label=f'False alarm cost  (${COST_FALSE_ALARM}/event)')
ax.plot(thresholds, fn_costs,   color='tomato',    lw=1.8,
        label=f'Missed defect cost (${COST_MISSED_DEFECT}/event)')
ax.plot(thresholds, total_costs, color='black',    lw=2.2, label='Total cost')
ax.axvline(best_t,      color='green',  ls='--', lw=1.8,
           label=f'Cost-optimal threshold = {best_t:.2f}')
ax.axvline(THRESHOLD_A3, color='orange', ls=':',  lw=1.8,
           label=f'Your threshold = {THRESHOLD_A3}')
ax.set_xlabel('Intervention Threshold', fontsize=11)
ax.set_ylabel('Estimated Cost ($)', fontsize=11)
ax.set_title('Cost Tradeoff vs Intervention Threshold — Adjust costs above to explore',
             fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.legend(fontsize=8.5)
plt.tight_layout(); plt.show()

idx = int(np.argmin(np.abs(thresholds - THRESHOLD_A3)))
print(f'At your threshold = {THRESHOLD_A3}:')
print(f'  False alarm cost:    ${fp_costs[idx]:>7,.0f}')
print(f'  Missed defect cost:  ${fn_costs[idx]:>7,.0f}')
print(f'  Total cost:          ${total_costs[idx]:>7,.0f}')
print(f'Cost-optimal threshold: {best_t:.2f}  (min total = ${min(total_costs):,.0f})')


### A3 · Cell 3 — Build Intervention Decisions

Fill in your decision for each flagged layer, then re-run.

**Actions and their effects:**
- `'continue'` — accept the risk; no cost but defect may form
- `'adjust'`   — update laser power/speed before that layer; costs \$400, reduces residual risk by 60%
- `'stop'`     — abort build entirely; costs \$15,000, eliminates all remaining risk


In [ ]:
# ┌─────────────────────────────────────────────────────────────────────┐
# │  DECISIONS: layer → 'continue' | 'adjust' | 'stop'                 │
# │  Modify the entries below to match your flagged layers from Cell 1  │
# └─────────────────────────────────────────────────────────────────────┘
DECISIONS = {
    23: 'adjust',
    24: 'adjust',
    25: 'continue',
    26: 'adjust',
    27: 'continue',
    34: 'adjust',
    35: 'adjust',
    36: 'stop',
}

ACTION_COLOR  = {'continue':'#3498db', 'adjust':'#f39c12', 'stop':'#e74c3c'}
ACTION_COST   = {'continue':0, 'adjust':COST_FALSE_ALARM, 'stop':15000}
ACTION_RISK_R = {'continue':0.0, 'adjust':0.60, 'stop':1.0}

fore_sorted = fore_d.sort_values('layer')
fig, ax = plt.subplots(figsize=(13,3.8))
for _, row in fore_sorted.iterrows():
    L      = int(row.layer)
    action = DECISIONS.get(L, 'continue')
    ax.bar(L, row.risk, color=ACTION_COLOR[action], edgecolor='white', width=0.7, alpha=0.85)
    if L in DECISIONS:
        ax.text(L, row.risk+0.04, action[:3].upper(),
                ha='center', fontsize=7.5, fontweight='bold', color=ACTION_COLOR[action])
ax.axhline(THRESHOLD_A3, color='red', lw=1.5, ls=':', label=f'Threshold = {THRESHOLD_A3}')
ax.set_xlabel('Layer'); ax.set_ylabel('Forecasted Risk Score')
ax.set_title('Your Build Control Decisions', fontweight='bold')
ax.set_ylim(0, 1.2)
ax.legend(handles=[mpatches.Patch(color=c, label=a) for a,c in ACTION_COLOR.items()],
          fontsize=8)
plt.tight_layout(); plt.show()

print('DECISION SUMMARY')
print('='*65)
total_cost = 0
for L, action in sorted(DECISIONS.items()):
    row = fore_d[fore_d.layer==L]
    if not len(row): continue
    risk  = float(row.risk.values[0])
    cost  = ACTION_COST[action]
    resid = risk * (1 - ACTION_RISK_R[action])
    total_cost += cost
    print(f'  Layer {L:2d}: {action.upper():<10}  '
          f'risk {risk:.2f} → residual {resid:.2f}   cost = ${cost:>6,}')
print(f'  {"─"*55}')
print(f'  Total intervention cost: ${total_cost:,.0f}')
print()
print('Reflection:')
print('  1. Which decision was hardest to justify? Why?')
print('  2. How does the cost-optimal threshold change if COST_MISSED_DEFECT doubles?')
print('  3. What additional data would make you more confident in the forecast?')
